# Native pre/post comparison with dendrite-filtered windows

Compare **mean**, **pointwise MLP**, and **GraphTransformer (GT)** on the native **bin16 / k17 / confidence ≥0.7** cohort. Each architecture is compared across pre only, a matched-width presynaptic-mean control, and pre + post, using the existing fold-0 cell splits and 30-epoch settings.

| Condition | Input to the shared ResNet classification trunk |
|---|---|
| Pre only | Pooled presynaptic representation |
| Pre + raw pre mean | Pooled presynaptic representation concatenated with the 64-dimensional mean of raw presynaptic embeddings |
| Pre + post | Pooled presynaptic representation concatenated with the matched 64-dimensional postsynaptic embedding |

Postsynaptic features enter **after pooling**, leaving the presynaptic encoder unchanged. All three groups exclude unresolved roots and empty postsynaptic masks before applying the same presynaptic-window deduplication. Different postsynaptic points **do not make identical presynaptic node sets unique**; the first eligible site supplies the postsynaptic vector.

The existing training procedure selects the best epoch using the held-out fold and reports metrics on that same fold. These are exploratory comparisons, not a separate untouched test set. Figures are separated by fold; missing runs are shown as pending.

The mean control and pre + post have identical classifier widths and parameter counts: **128 dimensions for mean**, **192 for MLP/GT**. The control adds no new observations, though it gives learned encoders a direct raw-feature skip connection. **Window macro F1 is the primary outcome**; cell-vote metrics are secondary context.
All conditions use the saved geodesic 20 µm soma-exclusion compartment masks. Windows with strictly more than half of observed embeddings predicting dendrite are removed; ties and other compartments remain. Filtering precedes postsynaptic eligibility and membership deduplication.

This experiment uses inverse-count sampling with replacement: each of the six L2 classes has an expected 1/6 share of training draws. All nine architecture/condition combinations are submitted.


In [ ]:
%load_ext autoreload
%autoreload 2
from pathlib import Path
import sys
import json
import pandas as pd
import matplotlib.pyplot as plt

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / 'scripts/train_gnn.py').is_file())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from analysis.presynaptic.native_pre_post_axon_filtered_equal_sampling.comparison import (
    RESULT_ROOT, ARCHITECTURES, ARCH_LABELS, CONDITIONS, LABELS,
    load_training_curves, load_summaries, metric_deltas,
    plot_training_curves, plot_metric_comparison, plot_per_class_recall, plot_confusions,
)

def show_fold_figure(fig, fold):
    if fig is not None:
        title = fig._suptitle.get_text() if fig._suptitle is not None else ''
        fig.suptitle(f'Fold {fold} · {title}')
        display(fig)
        plt.close(fig)

print(RESULT_ROOT)

## Training progress

Rows correspond to architectures; columns show training loss, held-out window macro F1, and held-out cell macro F1. Rerun the notebook to refresh progress from saved epoch CSV files.

In [ ]:
curves = load_training_curves()
if curves.empty:
    print('No completed epochs yet.')
else:
    for fold, fold_curves in curves.groupby('fold', sort=True):
        print(f'Fold {fold}')
        latest = (fold_curves.sort_values('epoch')
                  .groupby(['architecture', 'condition', 'run'], observed=True).tail(1))
        display(latest[['architecture', 'condition', 'epoch', 'train_loss',
                        'window_macro_f1', 'cell_macro_f1']].sort_values(['architecture', 'condition']))
        show_fold_figure(plot_training_curves(fold_curves), fold)

## Held-out metrics

Final result JSON files take precedence. Until a model finishes, its saved best-so-far metrics are shown with **complete=False**. Window macro F1 is the primary metric. Cell metrics reflect majority voting over retained presynaptic windows; window metrics measure local predictions.

In [ ]:
summary, payloads = load_summaries()
folds = sorted(summary.fold.unique()) if not summary.empty else []
if summary.empty:
    print('Awaiting saved best-epoch or final metrics.')
else:
    for fold in folds:
        fold_summary = summary[summary.fold == fold]
        display(fold_summary.drop(columns='run').sort_values(['architecture', 'condition']))
        expected = {(a, c) for a in ARCHITECTURES for c in CONDITIONS}
        present = set(zip(fold_summary.architecture, fold_summary.condition))
        print('Awaiting metrics:', sorted(expected - present) or 'none')
        show_fold_figure(plot_metric_comparison(fold_summary), fold)

## Benefit of postsynaptic information

Subtract each reference (**pre only** and **pre + raw pre mean**) from **pre + post**, within the same architecture and fold. The comparison against raw pre mean controls classifier input width and parameter count. Positive window macro F1 differences favor postsynaptic information. `both_complete=False` marks provisional comparisons.

In [ ]:
deltas = metric_deltas(summary)
if deltas.empty:
    print('Deltas require both conditions for the same architecture and fold.')
else:
    primary = ['fold', 'architecture', 'reference', 'both_complete', 'window_macro_f1',
               'window_balanced_accuracy', 'window_macro_precision', 'window_accuracy']
    display(deltas[primary])

## Per-class recall

In [ ]:
for fold in folds:
    for scope in ('cell', 'window'):
        show_fold_figure(plot_per_class_recall(payloads, fold, scope=scope), fold)

## Confusion matrices

One figure per architecture and fold. Window matrices are in the first row, cell matrices in the second. Each entry shows the fraction within its true class and the raw count in parentheses.

In [ ]:
for fold in folds:
    for architecture in ARCHITECTURES:
        show_fold_figure(plot_confusions(payloads, fold, architecture), fold)

## Excluded postsynaptic sites

The CSV lists the **25 cohort sites with empty postsynaptic masks**, including both root IDs and postsynaptic XYZ coordinates in nanometres. Unresolved roots are also excluded from training. Eligibility is shared by pre-only and pre+post models, and exclusions are applied before deduplication.

In [ ]:
excluded_path = ROOT / 'results/presynaptic/native_skeletons_pre_post/empty_mask_sites.csv'
excluded = pd.read_csv(excluded_path, dtype={
    'synapse_id': 'string', 'presynaptic_root_id': 'string', 'postsynaptic_root_id': 'string',
})
print(f'{len(excluded)} empty-mask sites; coordinates in nm')
display(excluded)
cache_marker = Path('/orcd/scratch/orcd/013/jcbliao/presynaptic_axons/native_skeletons_pre_post/scale16/k17/conf0.7/complete.json')
if cache_marker.exists():
    eligibility = json.loads(cache_marker.read_text())
    display(pd.Series({key: value for key, value in eligibility.items() if key.startswith('n_')}))

## Dendrite-window filtering audit
Counts below precede postsynaptic eligibility; training counts also reflect paired-site eligibility and deduplication.


In [ ]:
audit = pd.read_csv(ROOT / 'analysis/presynaptic/compartment_filtered/cell_audit.csv', dtype={'root_id': 'string'})
display(audit[audit.removed_sites > 0].sort_values('unique_removed', ascending=False))
display(audit[['valid_sites', 'removed_sites', 'unique_before', 'unique_removed', 'unique_retained']].sum())
